<!-- ## Diated Residual Networks -->
# 空洞残差网络

## 摘要 Abstract

卷积网络在图像分类中会逐步降低分辨率，直到图像被表示为微小的特征图，此时场景的空间结构已无法分辨。这种空间清晰度的损失会限制图像分类的准确性，并使模型迁移到需要详细场景理解的下游应用变得复杂。这些问题可以通过扩张操作来缓解，扩张操作在不减少单个神经元感受野的情况下提高了输出特征图的分辨率。我们证明，扩张残差网络（DRNs）在图像分类方面优于其非扩张的对应网络，且无需增加模型的深度或复杂度。然后，我们研究了扩张引入的网格伪影，开发了一种去除这些伪影的方法（‘去网格化’），并证明这进一步提高了 DRNs 的性能。此外，我们展示了 DRNs 在目标定位和语义分割等下游应用中的准确性优势被进一步放大。

## 空洞残差网络

卷积网络图像分类中，渐进式下采样虽适用于部分场景，但会丢失空间信息，不利于自然图像分类及相关任务迁移；而全程保留高空间分辨率并输出密集信号，可通过反向传播保留小物体等关键信息，提升分类效果。

$$\begin{aligned}

(\mathcal G^\ell_i \ast f^\ell_i)(\mathbf p) = \sum_{\mathbf a + \mathbf b = \mathbf p} \mathcal G^\ell_i(\mathbf a) \, f^\ell_i(\mathbf b),
\end{aligned}$$

直接去除网络深层的下采样虽能提升分辨率，但会成比例缩小后续层的感受野，损失关键上下文信息，得不偿失。因此，研究采用空洞卷积，在移除下采样以保留高分辨率的同时，补偿感受野的缩减，使空洞卷积层的单元感受野与原模型对应单元保持一致。

<div style="background-color:#FFFBCC; width:80%; padding:10px; border-radius:5px; margin: auto; text-align: center;">
    <image src="assets/resnet_groups.png" alt="ResNet" width="600"/>
    <div style="font-size: 0.9em; color: #555;">图1：ResNet的Groups<div>
</div>

<div style="background-color:#FFFBCC; width:80%; padding:10px; border-radius:5px; margin: auto; text-align: center;">
    <image src="assets/net_before_dilation.png" alt="Net Before Dilation" width="600"/>
    <div style="font-size: 0.9em; color: #555;">图2：在使用空洞卷积之前的网络结构示意图(ResNet)<div>
</div>

<div style="background-color:#FFFBCC; width:80%; padding:10px; border-radius:5px; margin: auto; text-align: center;">
    <image src="assets/net_after_dilation.png" alt="Net After Dilation" width="600"/>
    <div style="font-size: 0.9em; color: #555;">图3：在使用空洞卷积之后的网络结构示意图(DRN)<div>
</div>

## 定位 Localization

给定一个用于图像分类训练的 DRN，我们无需任何额外的训练或参数调整，可以直接生成密集的像素级类别激活图。这使得一个用于图像分类训练的 DRN 可以立即用于目标定位和分割。

<div style="background-color:#FFFBCC; width:80%; padding:10px; border-radius:5px; margin: auto; text-align: center;">
    <image src="assets/vis_original.png" alt="vis with pooling" width="600"/>
    <div style="font-size: 0.9em; color: #555;">图4: 原本的ResNet分类层: 全局平均池化 + 线性层<div>
</div>
<div style="background-color:#FFFBCC; width:80%; padding:10px; border-radius:5px; margin: auto; text-align: center;">
    <image src="assets/vis_no_pool.png" alt="vis without pooling" width="600"/>
    <div style="font-size: 0.9em; color: #555;">图5: 可以直接生成像素级类别激活图<div>
</div>

移除ResNet最后的全局平均池化，直接将线性层(1x1卷积)应用于最后的特征图，得到$n$个类别的高分辨率类激活图。

<style>
    /* 基础样式确保表格居中且图片响应式 */
    .figure-container {
        max-width: 100%;
        margin: 20px auto;
        text-align: center;
    }
    .image-table {
        width: 100%;
        border-collapse: separate;
        border-spacing: 8px; /* 替代LaTeX的hskip间距 */
        margin: 0 auto;
    }
    .image-table td {
        width: 19%; /* 保持原宽度比例 */
        vertical-align: middle;
        text-align: center;
    }
    .image-table img {
        width: 100%;
        height: auto;
        display: block;
    }
    .caption {
        margin-top: 15px;
        font-size: 14px;
        color: #ffffff;
        max-width: 1000px;
        margin-left: auto;
        margin-right: auto;
        text-align: left;
    }
    .label-row td {
        font-weight: bold;
        font-size: 13px;
        padding-top: 10px;
    }
    /* 响应式调整：小屏幕下减少间距 */
    @media (max-width: 768px) {
        .image-table {
            border-spacing: 4px;
        }
        .caption {
            font-size: 12px;
            padding: 0 10px;
        }
    }
</style>
<div class="figure-container">
    <table class="image-table">
        <!-- 第一行图片 -->
        <tr>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000021.JPEG" alt="Input 1"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000021_resnet18.png" alt="ResNet-18 1"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000021_drn18.png" alt="DRN-A-18 1"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000021_drn26_v2.png" alt="DRN-B-26 1"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000021_drn26_v3.png" alt="DRN-C-26 1"></td>
        </tr>
        <!-- 第二行图片 -->
        <tr>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000023.JPEG" alt="Input 2"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000023_resnet18.png" alt="ResNet-18 2"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000023_drn18.png" alt="DRN-A-18 2"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000023_drn26_v2.png" alt="DRN-B-26 2"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000023_drn26_v3.png" alt="DRN-C-26 2"></td>
        </tr>
        <!-- 第三行图片 -->
        <tr>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000109.JPEG" alt="Input 3"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000109_resnet18.png" alt="ResNet-18 3"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000109_drn18.png" alt="DRN-A-18 3"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000109_drn26_v2.png" alt="DRN-B-26 3"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000109_drn26_v3.png" alt="DRN-C-26 3"></td>
        </tr>
        <!-- 第四行图片 -->
        <tr>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000276.JPEG" alt="Input 4"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000276_resnet18.png" alt="ResNet-18 4"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000276_drn18.png" alt="DRN-A-18 4"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000276_drn26_v2.png" alt="DRN-B-26 4"></td>
            <td><img src="assets/drn_new/ILSVRC2012_val_00000276_drn26_v3.png" alt="DRN-C-26 4"></td>
        </tr>
        <!-- 标签行 -->
        <tr class="label-row">
            <td>(a) Input</td>
            <td>(b) ResNet-18</td>
            <td>(c) DRN-A-18</td>
            <td>(d) DRN-B-26</td>
            <td>(e) DRN-C-26</td>
        </tr>
    </table>
     <div class="caption">
        <strong>图 1：</strong>ResNet-18 及对应 DRN 模型的激活图。如第 3 节所述，基于 ResNet-18 构建的 DRN 称为 DRN-A-18；如第 4 节所述，通过解网格方案（degridding scheme）生成的对应 DRN 称为 DRN-C-26；DRN-B-26 为中间构建版本。
    </div>
</div>

## 重网格化 Degridding

<table style="background-color: white; width: 80%; text-align: center; margin: auto; padding: 20px; border-radius: 20px; border: none;">
    <tr>
        <td><image src="assets/gridding/before.png" alt="Gridding Artifact"/></td>
        <td><image src="assets/gridding/conv.png" alt="Gridding Artifact"/></td>
        <td><image src="assets/gridding/after.png" alt="Gridding Artifact"/></td>
    </tr>
    <tr class="label-row">
        <td>(a) Input</td>
        <td>(b) Dilation 2</td>
        <td>(c) Output</td>
    </tr>
</table>

使用扩张卷积可能会导致栅格伪影。这种伪影显示在图 3(c)中，并且在目前的语义分割工作中也已被观察到。当特征图的频率内容高于扩张卷积的采样率时，就会发生栅格伪影。图 4 展示了一个教学示例。在图 4(a)中，输入特征图只有一个激活像素。一个 2 扩张卷积（图 4(b)）在输出中诱导了相应的栅格模式（图 4(c)）。

<div style="background-color:#FFFBCC; width:70%; padding:10px; border-radius:5px; margin: auto; text-align: center;">
    <image src="assets/new_drn.png" alt="Net After Dilation" width="600"/>
    <div style="font-size: 0.9em; color: #555;">图5：改变 DRN 架构以消除输出激活图中的网格伪影。每个矩形是一个 Conv-BN-ReLU 组，数字指定该层的过滤器大小和通道数。粗绿线表示步长为 2 的下采样。网络分为多个层级，使得同一层级的所有层具有相同的扩张率和空间分辨率。<div>
</div>

1) DRN-A 直接扩张 ResNet 模型，如第 2 节所述。
2) DRN-B 用残差块替换了早期的一个最大池化层，并在网络的末端添加了残差块。
3) DRN-C 移除了一些添加块的残差连接。每一步的原理在文中进行了描述。

__移除最大池化 (Removing max pooling)__。如图 5(a)所示，DRN-A 在初始 $7 \times 7$卷积之后继承了 ResNet 架构中的最大池化操作。我们发现这个最大池化操作会导致高幅值高频激活，如图 6(b)所示。这种高频激活可以被传播到后续层，并最终加剧网格伪影。因此，我们用卷积滤波器替换最大池化，如图 5(b)所示。这种转换的效果如图 6(c)所示。



<div style="background-color: white; width: 80%; margin: auto; text-align: center; ">
    <table style="
        border: none;
        table-layout: fixed; /* 强制列宽均分，关键属性 */
    ">
        <tr>
            <!-- 每个单元格设置相同宽度，这里用百分比均分，5列则每列20% -->
            <td style="width: 20%; padding: 5px;"><img src="assets/max_pool/ILSVRC2012_val_00000077.jpg" alt="With Pooling" style="width: 100%; height: auto;"/></td>
            <td style="width: 20%; padding: 5px;"><img src="assets/max_pool/ILSVRC2012_val_00000077_drn18.png" alt="drn18" style="width: 100%; height: auto;"/></td>
            <td style="width: 20%; padding: 5px;"><img src="assets/max_pool/ILSVRC2012_val_00000077_drn26_v3.png" alt="drn26" style="width: 100%; height: auto;"/></td>
            <!-- <td style="width: 20%; padding: 5px;"><img src="assets/max_pool/ILSVRC2012_val_00000077_learned_sampling.png" alt="Learned Sampling" style="width: 100%; height: auto;"/></td> -->
            <!-- <td style="width: 20%; padding: 5px;"><img src="assets/max_pool/ILSVRC2012_val_00000077_max_pool.JPEG" alt="Max Pooling" style="width: 100%; height: auto;"/></td> -->
        </tr>
        <tr class="label-row">
            <!-- 标签行的单元格也对应设置相同宽度 -->
            <td style="width: 20%; padding: 5px;">(a) With Pooling</td>
            <td style="width: 20%; padding: 5px;">(b) drn18</td>
            <td style="width: 20%; padding: 5px;">(c) drn26</td>
            <!-- <td style="width: 20%; padding: 5px;">(d) Learned Sampling</td> -->
            <!-- <td style="width: 20%; padding: 5px;">(e) Max Pooling</td> -->
        </tr>
    </table>
    <span>
        <strong>图 6：</strong>去量化第一阶段，该阶段修改网络的前几层。（b）和（c）显示了 DRN-A-18 和 DRN-B-26 在 3 级的第一卷积层的输入特征图。显示了平均激活度最高的特征图。
    </span>
</div>

__增加层 Adding layers__。为了消除网格伪影，我们在网络的末端添加具有逐渐降低的膨胀率的卷积层。具体来说，在 DRN-A（图 5(a)）中最后一个 4 倍膨胀层之后，我们添加了一个 2 倍膨胀的残差块，然后是一个 1 倍膨胀的块。这些成为 DRN-B 中的第 7 层和第 8 层，如图 5(b)所示。这类似于使用具有适当频率的滤波器来消除混叠伪影[16]。

__移除残差连接 Removing residual connections__。按照前文所述，逐层增加具有递减扩张率的层，并不能完全消除网格伪影，因为存在残差连接。DRN-B 的第 7 层和第 8 层的残差连接会从第 6 层传播网格伪影。为了更有效地消除网格伪影，我们移除了 DRN-B 的第 7 层和第 8 层的残差连接。这产生了 DRN-C，即我们提出的结构，如图 5(c)所示。请注意，DRN-C 的深度和容量高于相应的 DRN-A 或作为起点的 ResNet。然而，我们将证明所提出的去网格方案对精度有显著影响，精度提升足以补偿增加的深度和容量。例如，实验将证明 DRN-C-26 在图像分类精度上与 DRN-A-34 相当，并且在目标定位和语义分割精度上高于 DRN-A-50。

DRN-C 内部的激活情况如图 7 所示。该图展示了网络每个层级的输出特征图。显示的是平均激活幅度最大的特征图。

<style>
.figure-container {
    max-width: 100%;
    margin: 20px auto;
    text-align: center;
}
.image-table {
    width: 100%;
    border-collapse: separate;
    border-spacing: 8px; /* 替代LaTeX的hskip间距 */
    margin: 0 auto;
}
.image-table td {
    width: 12.5%; /* 保持原宽度比例 */
    vertical-align: middle;
    text-align: center;
}
.image-table img {
    width: 100%;
    height: auto;
    display: block;
}
.caption {
    margin-top: 15px;
    font-size: 14px;
    color: #ffffff;
    max-width: 1000px;
    margin-left: auto;
    margin-right: auto;
    text-align: left;
}
.label-row td {
    font-weight: bold;
    font-size: 13px;
    padding-top: 10px;
}
</style>
<div class="figure-container">
    <table class="image-table">
        <!-- 第一行图片 -->
        <tr>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000003.JPEG" alt="Input"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000003_0.png" alt="Layer 0"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000003_1.png" alt="Layer 1"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000003_2.png" alt="Layer 2"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000003_3.png" alt="Layer 3"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000003_4.png" alt="Layer 4"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000003_5.png" alt="Layer 5"></td>
            <!-- <td><img src="assets/layer_activation/ILSVRC2012_val_00000003_6.png" alt="Layer 6"></td> -->
            <!-- <td><img src="assets/layer_activation/ILSVRC2012_val_00000003_7.png" alt="Layer 7"></td> -->
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000003_final.png" alt="Final Layer"></td>
        </tr>
        <!-- 第二行图片 -->
        <tr>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000014.JPEG" alt="Input"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000014_0.png" alt="Layer 0"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000014_1.png" alt="Layer 1"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000014_2.png" alt="Layer 2"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000014_3.png" alt="Layer 3"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000014_4.png" alt="Layer 4"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000014_5.png" alt="Layer 5"></td>
            <!-- <td><img src="assets/layer_activation/ILSVRC2012_val_00000014_6.png" alt="Layer 6"></td> -->
            <!-- <td><img src="assets/layer_activation/ILSVRC2012_val_00000014_7.png" alt="Layer 7"></td> -->
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000014_final.png" alt="Final Layer"></td>
        </tr>
        <!-- 第三行图片 -->
        <tr>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000283.JPEG" alt="Input"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000283_0.png" alt="Layer 0"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000283_1.png" alt="Layer 1"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000283_2.png" alt="Layer 2"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000283_3.png" alt="Layer 3"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000283_4.png" alt="Layer 4"></td>
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000283_5.png" alt="Layer 5"></td>
            <!-- <td><img src="assets/layer_activation/ILSVRC2012_val_00000283_6.png" alt="Layer 6"></td> -->
            <!-- <td><img src="assets/layer_activation/ILSVRC2012_val_00000283_7.png" alt="Layer 7"></td> -->
            <td><img src="assets/layer_activation/ILSVRC2012_val_00000283_final.png" alt="Final Layer"></td>
        </tr>
        <!-- 标签行 -->
        <tr class="label-row">
            <td>Image</td>
            <td>Level 1</td>
            <td>Level 2</td>
            <td>Level 3</td>
            <td>Level 4</td>
            <td>Level 5</td>
            <td>Level 6</td>
            <td>Class activation</td>
        </tr>
    </table>
     <div class="caption">
        <strong>图 7：</strong>训练好的 DRN-C-26 内部的激活情况。对于每个级别，我们展示了该级别输出中平均激活幅度最高的特征图。这些级别在图 5 中定义。
    </div>
</div>